In [7]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/RJ_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  0000000000000000090c_0 -0.046207       1  0.235760  0.090506  0.390437   
1  0000000000000000090c_0 -0.040776       1  0.221084  0.081945  0.378913   
2  0000000000000000090c_0 -0.122922       1  0.343448  0.195024  0.451904   
3  0000000000000000097b_0 -0.063690       1  0.216320  0.117444  0.368708   
4  0000000000000000090c_0 -0.050548       1  0.204699  0.105839  0.350062   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  74.355802   
1  2021  {"geodesic":false,"type":"Point","coordinates"...  74.355802   
2  2022  {"geodesic":false,"type":"Point","coordinates"...  74.355802   
3  2023  {"geodesic":false,"type":"Point","coordinates"...  74.355802   
4  2023  {"geodesic":false,"type":"Point","coordinates"...  74.355802   

    latitude  
0  23.476122  
1  23.476122  
2  23.476122  
3  23.476122  
4  23.476122  
[2021 20

In [9]:
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

Raw input shape: (3994, 4, 5)


In [10]:

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/RJ_Deforestation_Predictions.csv',
    index=False
)

print("Saved:RJ_Deforestation_Predictions.csv")



Label distribution:
No deforestation: 3465
Deforestation: 529
Train shape: (3195, 4, 5)
Test shape : (799, 4, 5)
Scaled train shape: (3195, 4, 5)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.1525 - loss: 1.6590 - precision: 0.1350 - recall: 0.9819 - val_accuracy: 0.1327 - val_loss: 0.9587 - val_precision: 0.1327 - val_recall: 1.0000
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1299 - loss: 1.5729 - precision: 0.1299 - recall: 1.0000 - val_accuracy: 0.1327 - val_loss: 0.9289 - val_precision: 0.1327 - val_recall: 1.0000
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1317 - loss: 1.5183 - precision: 0.1299 - recall: 0.9980 - val_accuracy: 0.2353 - val_loss: 0.8837 - val_precision: 0.1469 - val_recall: 0.9906
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2721 - loss: 1.4291 - precision: 0.1542 - recall: 0.9628 - val_accuracy: 0.4856 - val_loss: 0.7654 - val_precision: 0.1932 - val_recall: 0.9057
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4980 - loss: 1.2714 - precision: 0.1953 - recall: 0.8866 - val_accuracy: 0.5482 - val_loss: 0.7268 

In [11]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/RJ_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/RJ_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/RJ_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.05954872611875313), 'std': 0.0950545421266009, 'q1': np.float64(-0.11324191749999998), 'q3': np.float64(-0.0067119925000000275), 'lower_2std': np.float64(-0.24965781037195495), 'upper_2std': np.float64(0.13056035813444866)}
NBR : {'mean': np.float64(-0.03651722776000351), 'std': 0.1119129905295288, 'q1': np.float64(-0.09992767799999999), 'q3': np.float64(0.021092226499999995), 'lower_2std': np.float64(-0.2603432088190611), 'upper_2std': np.float64(0.18730875329905408)}
BSI : {'mean': np.float64(0.0054244563360713515), 'std': 0.07271749339312172, 'q1': np.float64(-0.03152925635834908), 'q3': np.float64(0.046531893248393574), 'lower_2std': np.float64(-0.1400105304501721), 'upper_2std': np.float64(0.1508594431223148)}
NDMI: {'mean': np.float64(-0.012688473242289933), 'std': 0.08406783917953395, 'q1': np.float64(-0.06092966425), 'q3': np.float64(0.029853599000000012), 'lower_2std': np.float64(-0.18082415160135784), 'upper_2std': np.float64(0.15544720511677795)}

In [13]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/RJ_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Urban/Other    152
Unknown         11
Name: count, dtype: int64


In [14]:
m.save('/content/drive/MyDrive/rj_adaptive.html')


In [15]:
m

In [16]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/RJ_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/RJ_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD RAJASTHAN DISTRICTS
# ==============================

rj = gpd.read_file('/content/drive/MyDrive/RJ.geojson')

# CRS SAFETY
if rj.crs is None:
    rj.set_crs("EPSG:4326", inplace=True)

rj = rj.to_crs("EPSG:4326")

# Add state label (optional)
rj["state"] = "Rajasthan"

print(rj.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 799
    latitude  longitude  deforestation
0  25.217057  72.130675              1
1  29.783194  74.334242              0
2  24.642136  72.595104              0
3  29.819127  74.243512              0
4  27.151130  77.532245              0
Total deforestation samples: 163
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  25.217057  72.130675              1     0.029069    0.082060   -0.064736   
1  27.148435  72.964311              1     0.003890    0.072043   -0.055589   
2  24.660102  72.628342              1     0.005835    0.082899   -0.081771   
3  24.338505  74.426769              1     0.000455    0.078518   -0.051650   
4  26.261798  73.068516              1     0.047015    0.051019   -0.037498   

   NDMI_change        cause  
0     0.057221  Urban/Other  
1     0.080567  Urban/Other  
2     0.094245  Urban/Other  
3     0.075327  Urban/Other  
4     0.038688  Urban/Other  
  REMARKS_2 Country State_Name State_Code  Dist_Name Dist_C

In [17]:
# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    rj,                # Rajasthan GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — RJ
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/RJ_District_Wise_Deforestation.csv',
    index=False
)

print("Saved RJ district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/RJ_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved RJ cause summary")

    latitude  longitude  deforestation        cause  \
0  25.217057  72.130675              1  Urban/Other   
1  27.148435  72.964311              1  Urban/Other   
2  24.660102  72.628342              1  Urban/Other   
3  24.338505  74.426769              1  Urban/Other   
4  26.261798  73.068516              1  Urban/Other   

                    geometry  index_right REMARKS_2 Country State_Name  \
0  POINT (72.13067 25.21706)           25      None   India  Rajasthan   
1  POINT (72.96431 27.14844)           28      None   India  Rajasthan   
2   POINT (72.62834 24.6601)            7      None   India  Rajasthan   
3  POINT (74.42677 24.33851)           32      None   India  Rajasthan   
4   POINT (73.06852 26.2618)           28      None   India  Rajasthan   

  State_Code     Dist_Name Dist_Code      state  
0         08         Jalor       116  Rajasthan  
1         08       Jodhpur       113  Rajasthan  
2         08        Sirohi       117  Rajasthan  
3         08  Chittaurga

In [18]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Rajasthan District Boundaries
# =====================================================
rj = gpd.read_file('/content/drive/MyDrive/RJ.geojson')

# CRS safety
if rj.crs is None:
    rj.set_crs(epsg=4326, inplace=True)

rj = rj.to_crs(epsg=4326)
rj["state"] = "Rajasthan"
gdf_districts = rj.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/RJ_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0  # placeholder if no data

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join (assign districts to points)
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats into District Map
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="Dist_Name", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)
gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)
gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)
gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)
gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Rajasthan)
# =====================================================
m = folium.Map(location=[27.0, 74.5], zoom_start=6)

# =====================================================
# STEP 10: Dynamic Choropleth for Deforestation
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Rajasthan)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Rajasthan Afforestation Alert Popup
# =====================================================

state_total = gdf_districts["total_samples"].sum()
state_def = gdf_districts["deforestation_samples"].sum()
state_aff = gdf_districts["afforestation_samples"].sum()

# Get Top 3 districts by deforestation
top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

# Build dynamic district list
top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Rajasthan Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on plantation, controlled logging,<br>
and sustainable land management.
"""

folium.Marker(
    location=[27.0, 74.5],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)
# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/rj_forest.html")

print("✅ Rajasthan map saved successfully with afforestation recommendation!")

/tmp/ipython-input-157/200153076.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Rajasthan map saved successfully with afforestation recommendation!
